# 01 · Evaluación de modelos preentrenados

**Qué hace:** pipeline LaMa (inpainting) + A-ESRGAN (super-res) preentrenados sobre una imagen degradada, con métricas full-reference PSNR/SSIM/LPIPS.

**Qué necesita:**
- Kaggle: `marcinrutecki/old-photos`, `shrutimandaokar2301/vintage-degraded-image-synthetic-real` (se descargan en la primera celda)
- GPU (Google Colab)

**Qué deja escrito:** figuras y una tabla LPIPS en `ROOT/_out/01/` (nada en Drive)

**Arranque:** primera celda de código.

## 0. Instalación básica

Instala las dependencias comunes del notebook. Las dependencias específicas de A-ESRGAN se instalan más adelante, después de clonar el repositorio.

> Si el runtime ya había importado `PIL/Pillow` antes de instalar paquetes, reinicia el kernel/runtime y ejecuta el notebook desde el principio. En notebooks es frecuente que una instalación parcial de `Pillow` deje módulos antiguos y nuevos cargados al mismo tiempo.


In [ ]:
!pip -q install kagglehub simple-lama-inpainting lpips scikit-image pandas
from colab_setup import aplicar_parches, preparar_repo, descargar_datos
aplicar_parches()
ROOT = preparar_repo()
DATA = descargar_datos("kaggle:old-photos", "kaggle:vintage-degraded", root=ROOT)

In [ ]:
%matplotlib inline

from pathlib import Path
import os
import sys
import site
import subprocess
import shutil
from typing import Optional, Sequence

import cv2
import numpy as np
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
from IPython.display import display

# Permite leer algunas imágenes antiguas parcialmente truncadas.
ImageFile.LOAD_TRUNCATED_IMAGES = True

try:
    from google.colab import output
    IN_COLAB = True
except Exception:
    output = None
    IN_COLAB = False

IMAGE_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tif', '.tiff')

WORK_DIR = ROOT / '_out' / '01'
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f'Entorno Colab: {IN_COLAB}')
print(f'Directorio de trabajo: {WORK_DIR}')
print(f'Pillow: {Image.__version__}')
print(f'OpenCV: {cv2.__version__}')

## 1. Descarga y exploración de datasets

Se descargan los dos datasets usados en esta fase de prueba:

- `Old Photos`: imágenes antiguas reales.
- `Vintage Degraded Image: Synthetic + Real`: imágenes limpias, dañadas y degradadas sintéticamente.

In [ ]:
old_photos_path = DATA['kaggle:old-photos']
vintage_photos_path = DATA['kaggle:vintage-degraded']

vintage_clean_path = vintage_photos_path / '01_Clean_Candidates_GT'
vintage_damaged_path = vintage_photos_path / '02_Damaged_Testing_Set'
synthetic_damaged_path = vintage_photos_path / '03_Synthetic_Dataset' / 'Train_Input_Degraded'

DATASET_DIRS = {
    'Old Photos': old_photos_path,
    'Vintage clean': vintage_clean_path,
    'Vintage damaged': vintage_damaged_path,
    'Synthetic degraded': synthetic_damaged_path,
}

for name, path in DATASET_DIRS.items():
    print(f'{name:20s}: {path}')
    if not path.exists():
        raise FileNotFoundError(f'No existe la ruta esperada: {path}')

In [ ]:
def list_images(root: Path, recursive: bool = False) -> list[Path]:
    """Devuelve imágenes bajo `root`, ordenadas alfabéticamente."""
    root = Path(root)
    pattern = '**/*' if recursive else '*'
    return sorted(
        p for p in root.glob(pattern)
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )


def _read_image_with_cv2(path: Path, flags: int) -> np.ndarray:
    """Lee una imagen con OpenCV evitando `PIL.Image.open()` para JPEG.

    Esta función evita el error:
    AttributeError: property 'mode' of 'JpegImageFile' object has no setter
    que puede aparecer cuando el runtime mezcla versiones de Pillow.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    data = np.fromfile(str(path), dtype=np.uint8)
    image = cv2.imdecode(data, flags)
    if image is None:
        raise ValueError(f'OpenCV no pudo leer la imagen: {path}')
    return image


def open_rgb(path: Path) -> Image.Image:
    """Abre una imagen desde disco como PIL RGB usando OpenCV para la decodificación."""
    image = _read_image_with_cv2(path, cv2.IMREAD_UNCHANGED)

    if image.ndim == 2:
        image_rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        return Image.fromarray(image_rgb)

    if image.ndim != 3:
        raise ValueError(f'Formato de imagen no soportado: {path} con shape={image.shape}')

    channels = image.shape[2]
    if channels == 3:
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return Image.fromarray(image_rgb)

    if channels == 4:
        image_rgba = cv2.cvtColor(image, cv2.COLOR_BGRA2RGBA)
        return Image.fromarray(image_rgba).convert('RGB')

    raise ValueError(f'Número de canales no soportado: {channels} en {path}')


def open_grayscale(path: Path) -> Image.Image:
    """Abre una imagen desde disco como PIL L usando OpenCV para la decodificación."""
    image_gray = _read_image_with_cv2(path, cv2.IMREAD_GRAYSCALE)
    return Image.fromarray(image_gray)


def show_image(image: Image.Image, title: str = '', figsize: tuple[int, int] = (8, 6)) -> None:
    plt.figure(figsize=figsize)
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def show_image_grid(paths: Sequence[Path], title: str = '', max_images: int = 5, columns: int = 5) -> None:
    paths = list(paths)[:max_images]
    if not paths:
        print(f'No hay imágenes para mostrar: {title}')
        return

    rows = int(np.ceil(len(paths) / columns))
    plt.figure(figsize=(4 * columns, 4 * rows))

    for i, path in enumerate(paths, start=1):
        plt.subplot(rows, columns, i)
        try:
            plt.imshow(open_rgb(path))
            plt.title(path.name, fontsize=8)
        except Exception as exc:
            plt.text(0.5, 0.5, f'No se pudo leer\n{path.name}\n{type(exc).__name__}', ha='center', va='center')
            print(f'No se pudo leer {path}: {exc}')
        plt.axis('off')

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def show_stage_comparison(items: Sequence[tuple[str, Image.Image]], figsize: tuple[int, int] = (20, 6)) -> None:
    plt.figure(figsize=figsize)
    for i, (title, image) in enumerate(items, start=1):
        plt.subplot(1, len(items), i)
        if image.mode == 'L':
            plt.imshow(image, cmap='gray')
        else:
            plt.imshow(image)
        plt.title(title)
        plt.axis('off')
    plt.tight_layout()
    plt.show()


def pil_rgb_to_array(image: Image.Image, size: Optional[tuple[int, int]] = None) -> np.ndarray:
    """Convierte una imagen PIL RGB a array float32 [0, 255], opcionalmente reescalando."""
    image = image.convert('RGB')
    if size is not None and image.size != size:
        image = image.resize(size, Image.Resampling.LANCZOS)
    return np.asarray(image).astype(np.float32)


def pil_mask_to_array(mask: Image.Image, size: tuple[int, int]) -> np.ndarray:
    """Convierte una máscara PIL a array booleano, reescalando si es necesario."""
    mask = mask.convert('L')
    if mask.size != size:
        mask = mask.resize(size, Image.Resampling.NEAREST)
    return np.asarray(mask) > 127

In [ ]:
for name, path in DATASET_DIRS.items():
    files = list_images(path)
    print(f'{name:20s}: {len(files):4d} imágenes')
    show_image_grid(files, title=name, max_images=5, columns=5)

## 2. Selección de imagen de prueba

La variable `INPUT_IMAGE_PATH` fija la imagen de entrada. Si el archivo concreto no existe en el dataset descargado, se usa la primera imagen disponible en el directorio sintético.

In [ ]:
DEFAULT_TEST_IMAGE = synthetic_damaged_path / 'lrp_img52_(Complex_All).jpg'

if DEFAULT_TEST_IMAGE.exists():
    INPUT_IMAGE_PATH = DEFAULT_TEST_IMAGE
else:
    synthetic_files = list_images(synthetic_damaged_path)
    if not synthetic_files:
        raise FileNotFoundError(f'No hay imágenes en {synthetic_damaged_path}')
    INPUT_IMAGE_PATH = synthetic_files[0]

print(f'Imagen seleccionada: {INPUT_IMAGE_PATH}')
test_img = open_rgb(INPUT_IMAGE_PATH)
show_image(test_img, title=f'Entrada: {INPUT_IMAGE_PATH.name}', figsize=(9, 7))

## 3. Creación de máscara

LaMa necesita una máscara binaria de un canal: los píxeles blancos (`255`) indican la zona que se debe reconstruir.

Modos disponibles:

- `auto_rectangle`: crea una máscara rectangular reproducible para pruebas rápidas.
- `manual`: abre un editor de máscara en Colab.
- `file`: carga una máscara existente desde disco.

Para una prueba visual rápida se usa `auto_rectangle`. Para evaluar daños reales de fotografías antiguas, cambia `MASK_MODE` a `manual` o `file`.

In [ ]:
MASK_MODE = 'auto_rectangle'  # opciones: 'auto_rectangle', 'manual', 'file'
MASK_FILE = None              # ruta de máscara si MASK_MODE = 'file'
AUTO_MASK_BOX_FRACTION = (0.35, 0.25, 0.65, 0.55)  # left, top, right, bottom en proporción de la imagen

_MASK_EDITOR = {'data_url': None, 'original_size': None, 'display_size': None}


def pil_to_base64_png(image: Image.Image) -> str:
    import base64
    import io

    buffer = io.BytesIO()
    image.save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode('utf-8')


def _receive_mask_from_js(mask_data_url: str) -> None:
    global _MASK_EDITOR
    _MASK_EDITOR['data_url'] = mask_data_url
    print('Máscara recibida. Ejecuta la celda de construcción de máscara.')


if IN_COLAB:
    output.register_callback('notebook.receive_mask', _receive_mask_from_js)


def show_mask_editor(image: Image.Image, max_display_size: int = 900) -> None:
    """Muestra un editor HTML/JS para dibujar una máscara en Google Colab."""
    if not IN_COLAB:
        raise RuntimeError('El editor manual usa callbacks de Google Colab. Usa MASK_MODE="file" o "auto_rectangle" en local.')

    from IPython.display import HTML, display

    global _MASK_EDITOR
    _MASK_EDITOR['data_url'] = None

    image_rgb = image.convert('RGB')
    width, height = image_rgb.size
    scale = min(max_display_size / width, max_display_size / height, 1.0)
    display_width = int(width * scale)
    display_height = int(height * scale)

    _MASK_EDITOR['original_size'] = (width, height)
    _MASK_EDITOR['display_size'] = (display_width, display_height)

    display_image = image_rgb.resize((display_width, display_height))
    image_b64 = pil_to_base64_png(display_image)

    html = f"""
    <div id="mask-editor-container" style="font-family: sans-serif;">
      <div style="margin-bottom: 10px;">
        <span>Grosor del pincel: </span>
        <input id="brush-slider" type="range" min="1" max="100" value="25">
        <span id="brush-value">25 px</span>
        <button id="draw-button" style="margin-left: 10px;">Dibujar</button>
        <button id="erase-button" style="margin-left: 5px;">Borrar</button>
        <button id="clear-button" style="margin-left: 5px;">Limpiar</button>
        <button id="done-button" style="margin-left: 10px; font-weight: bold;">Usar máscara</button>
      </div>

      <div style="position: relative; width: {display_width}px; height: {display_height}px; border: 1px solid #999;">
        <canvas id="bg-canvas" width="{display_width}" height="{display_height}" style="position:absolute; left:0; top:0;"></canvas>
        <canvas id="draw-canvas" width="{display_width}" height="{display_height}" style="position:absolute; left:0; top:0; cursor:crosshair;"></canvas>
      </div>

      <p id="mask-status" style="font-weight: bold; color: #555;">Dibuja sobre la imagen y pulsa “Usar máscara”.</p>
    </div>

    <script>
      const imgWidth = {display_width};
      const imgHeight = {display_height};
      const bgCanvas = document.getElementById('bg-canvas');
      const drawCanvas = document.getElementById('draw-canvas');
      const bgCtx = bgCanvas.getContext('2d');
      const drawCtx = drawCanvas.getContext('2d');
      const brushSlider = document.getElementById('brush-slider');
      const brushValue = document.getElementById('brush-value');
      const drawButton = document.getElementById('draw-button');
      const eraseButton = document.getElementById('erase-button');
      const clearButton = document.getElementById('clear-button');
      const doneButton = document.getElementById('done-button');
      const statusText = document.getElementById('mask-status');
      let drawing = false;
      let mode = 'draw';

      const image = new Image();
      image.onload = () => bgCtx.drawImage(image, 0, 0, imgWidth, imgHeight);
      image.src = 'data:image/png;base64,{image_b64}';

      brushSlider.oninput = () => brushValue.innerText = brushSlider.value + ' px';
      drawButton.onclick = () => {{ mode = 'draw'; statusText.innerText = 'Modo dibujo.'; }};
      eraseButton.onclick = () => {{ mode = 'erase'; statusText.innerText = 'Modo borrado.'; }};
      clearButton.onclick = () => {{ drawCtx.clearRect(0, 0, imgWidth, imgHeight); statusText.innerText = 'Máscara limpiada.'; }};

      function getPos(e) {{
        const rect = drawCanvas.getBoundingClientRect();
        return {{ x: e.clientX - rect.left, y: e.clientY - rect.top }};
      }}

      function paint(e) {{
        if (!drawing) return;
        const pos = getPos(e);
        drawCtx.lineWidth = Number(brushSlider.value);
        drawCtx.lineCap = 'round';
        drawCtx.lineJoin = 'round';
        if (mode === 'draw') {{
          drawCtx.globalCompositeOperation = 'source-over';
          drawCtx.strokeStyle = 'rgba(255, 0, 0, 0.65)';
        }} else {{
          drawCtx.globalCompositeOperation = 'destination-out';
        }}
        drawCtx.lineTo(pos.x, pos.y);
        drawCtx.stroke();
        drawCtx.beginPath();
        drawCtx.moveTo(pos.x, pos.y);
      }}

      drawCanvas.addEventListener('mousedown', e => {{ drawing = true; const pos = getPos(e); drawCtx.beginPath(); drawCtx.moveTo(pos.x, pos.y); paint(e); }});
      drawCanvas.addEventListener('mousemove', paint);
      drawCanvas.addEventListener('mouseup', () => {{ drawing = false; drawCtx.beginPath(); }});
      drawCanvas.addEventListener('mouseleave', () => {{ drawing = false; drawCtx.beginPath(); }});

      doneButton.onclick = async () => {{
        const maskCanvas = document.createElement('canvas');
        maskCanvas.width = imgWidth;
        maskCanvas.height = imgHeight;
        const maskCtx = maskCanvas.getContext('2d');
        const drawnData = drawCtx.getImageData(0, 0, imgWidth, imgHeight).data;
        const maskImageData = maskCtx.createImageData(imgWidth, imgHeight);
        const maskData = maskImageData.data;
        for (let i = 0; i < drawnData.length; i += 4) {{
          const alpha = drawnData[i + 3];
          const value = alpha > 0 ? 255 : 0;
          maskData[i] = value;
          maskData[i + 1] = value;
          maskData[i + 2] = value;
          maskData[i + 3] = 255;
        }}
        maskCtx.putImageData(maskImageData, 0, 0);
        const maskDataURL = maskCanvas.toDataURL('image/png');
        statusText.innerText = 'Enviando máscara a Python...';
        statusText.style.color = 'orange';
        await google.colab.kernel.invokeFunction('notebook.receive_mask', [maskDataURL], {{}});
        statusText.innerText = 'Máscara guardada. Ejecuta la siguiente celda.';
        statusText.style.color = 'green';
      }};
    </script>
    """

    display(HTML(html))


def get_drawn_mask(image: Image.Image) -> Image.Image:
    """Recupera la máscara dibujada en el editor manual."""
    if _MASK_EDITOR['data_url'] is None:
        raise ValueError('No hay máscara guardada. Dibuja sobre la imagen y pulsa “Usar máscara”.')

    import base64

    mask_b64 = _MASK_EDITOR['data_url'].split(',', 1)[1]
    mask_buffer = np.frombuffer(base64.b64decode(mask_b64), dtype=np.uint8)
    mask_np = cv2.imdecode(mask_buffer, cv2.IMREAD_GRAYSCALE)
    if mask_np is None:
        raise ValueError('No se pudo decodificar la máscara recibida desde el editor manual.')

    mask_small = Image.fromarray(mask_np)
    original_size = _MASK_EDITOR['original_size']
    display_size = _MASK_EDITOR['display_size']

    if display_size != original_size:
        mask = mask_small.resize(original_size, Image.Resampling.NEAREST)
    else:
        mask = mask_small

    return binarize_mask(mask)


def binarize_mask(mask: Image.Image, size: Optional[tuple[int, int]] = None, threshold: int = 127) -> Image.Image:
    """Convierte una máscara a binaria: 0 fondo, 255 zona a reconstruir."""
    mask = mask.convert('L')
    if size is not None and mask.size != size:
        mask = mask.resize(size, Image.Resampling.NEAREST)
    mask_array = np.array(mask)
    mask_array = np.where(mask_array > threshold, 255, 0).astype(np.uint8)
    return Image.fromarray(mask_array, mode='L')


def create_rectangle_mask(size: tuple[int, int], box_fraction: tuple[float, float, float, float]) -> Image.Image:
    """Crea una máscara rectangular usando coordenadas relativas: left, top, right, bottom."""
    width, height = size
    left, top, right, bottom = box_fraction
    x1, y1 = int(left * width), int(top * height)
    x2, y2 = int(right * width), int(bottom * height)
    mask = np.zeros((height, width), dtype=np.uint8)
    mask[y1:y2, x1:x2] = 255
    return Image.fromarray(mask, mode='L')


def load_mask(mask_path: Path, size: tuple[int, int]) -> Image.Image:
    if mask_path is None:
        raise ValueError('MASK_FILE no puede ser None cuando MASK_MODE="file".')
    mask_path = Path(mask_path)
    if not mask_path.exists():
        raise FileNotFoundError(mask_path)
    return binarize_mask(open_grayscale(mask_path), size=size)


def apply_mask(image: Image.Image, mask: Image.Image, fill_value: int = 0) -> Image.Image:
    """Visualización auxiliar: pinta en negro la zona marcada por la máscara."""
    image_np = np.array(image.convert('RGB')).copy()
    mask_np = np.array(mask.convert('L'))
    image_np[mask_np == 255] = fill_value
    return Image.fromarray(image_np)

In [ ]:
if MASK_MODE == 'manual':
    show_mask_editor(test_img)
else:
    print(f'MASK_MODE={MASK_MODE!r}; no se abre el editor manual.')

In [ ]:
if MASK_MODE == 'manual':
    test_mask = get_drawn_mask(test_img)
elif MASK_MODE == 'file':
    test_mask = load_mask(MASK_FILE, size=test_img.size)
elif MASK_MODE == 'auto_rectangle':
    test_mask = create_rectangle_mask(test_img.size, AUTO_MASK_BOX_FRACTION)
else:
    raise ValueError(f'MASK_MODE no válido: {MASK_MODE}')

image_masked = apply_mask(test_img, test_mask)

show_stage_comparison([
    ('Imagen original/degradada', test_img),
    ('Máscara binaria', test_mask),
    ('Visualización de entrada enmascarada', image_masked),
], figsize=(18, 5))

## 4. Inpainting con LaMa

LaMa se aplica sobre la imagen original y la máscara binaria. La imagen enmascarada anterior es solo una visualización para verificar que la máscara cubre la zona correcta.

In [ ]:
from simple_lama_inpainting import SimpleLama

simple_lama = SimpleLama()
lama_result = simple_lama(test_img, test_mask).convert('RGB')

ORIGINAL_PATH = WORK_DIR / '01_original.png'
MASK_PATH = WORK_DIR / '02_mask.png'
MASKED_PATH = WORK_DIR / '03_masked_preview.png'
LAMA_OUTPUT_PATH = WORK_DIR / '04_lama_inpainting.png'

test_img.save(ORIGINAL_PATH)
test_mask.save(MASK_PATH)
image_masked.save(MASKED_PATH)
lama_result.save(LAMA_OUTPUT_PATH)

print(f'Resultado LaMa guardado en: {LAMA_OUTPUT_PATH}')
show_stage_comparison([
    ('Entrada', test_img),
    ('Máscara', test_mask),
    ('LaMa inpainting', lama_result),
], figsize=(18, 5))

## 5. Instalación de A-ESRGAN

Se clona el repositorio, se instalan sus dependencias y se descarga el peso preentrenado `A_ESRGAN_Single.pth`.

La celda también parchea una importación antigua de `basicsr` que puede fallar con versiones recientes de `torchvision`.

In [ ]:
AESRGAN_REPO_URL = 'https://github.com/stroking-fishes-ml-corp/A-ESRGAN.git'
AESRGAN_DIR = Path('/content/A-ESRGAN') if IN_COLAB else Path.cwd() / 'A-ESRGAN'
AESRGAN_MODEL_URL = 'https://github.com/stroking-fishes-ml-corp/A-ESRGAN/releases/download/v1.0.0/A_ESRGAN_Single.pth'
AESRGAN_MODEL_PATH = AESRGAN_DIR / 'experiments' / 'pretrained_models' / 'A_ESRGAN_Single.pth'

if not AESRGAN_DIR.exists():
    subprocess.run(['git', 'clone', AESRGAN_REPO_URL, str(AESRGAN_DIR)], check=True)
else:
    print(f'Repositorio ya disponible: {AESRGAN_DIR}')
    subprocess.run(['git', '-C', str(AESRGAN_DIR), 'pull'], check=True)

# Importante: las especificaciones de versión van entre comillas para evitar redirecciones de shell con ">=".
# No forzamos la reinstalación de Pillow para evitar inconsistencias en kernels ya iniciados.
%pip install -q --upgrade-strategy only-if-needed "basicsr>=1.3.3.11" "facexlib>=0.2.0.3" "gfpgan>=0.2.1" opencv-python tqdm

AESRGAN_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
if not AESRGAN_MODEL_PATH.exists():
    subprocess.run([
        'wget', '-q', '-O', str(AESRGAN_MODEL_PATH), AESRGAN_MODEL_URL
    ], check=True)
else:
    print(f'Pesos ya disponibles: {AESRGAN_MODEL_PATH}')

print(f'A-ESRGAN dir: {AESRGAN_DIR}')
print(f'Modelo: {AESRGAN_MODEL_PATH}')

In [ ]:
def patch_basicsr_rgb_to_grayscale_import() -> None:
    """Corrige la importación de rgb_to_grayscale en basicsr si usa la ruta antigua."""
    candidates = []
    for base in site.getsitepackages() + [site.getusersitepackages()]:
        candidates.append(Path(base) / 'basicsr' / 'data' / 'degradations.py')

    patched = False
    for path in candidates:
        if not path.exists():
            continue
        text = path.read_text(encoding='utf-8')
        old = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
        new = 'from torchvision.transforms.functional import rgb_to_grayscale'
        if old in text:
            path.write_text(text.replace(old, new), encoding='utf-8')
            print(f'Parche aplicado en: {path}')
            patched = True
        else:
            print(f'No requiere parche: {path}')
            patched = True
    if not patched:
        print('No se encontró basicsr/data/degradations.py. Verifica la instalación de basicsr.')


patch_basicsr_rgb_to_grayscale_import()

## 6. Super-resolución con A-ESRGAN

La entrada de A-ESRGAN es la salida de LaMa. Se usa `--tile` para reducir memoria en GPU. Desactiva `USE_HALF_PRECISION` si tu entorno no soporta inferencia en media precisión.

In [ ]:
AESRGAN_OUTPUT_DIR = WORK_DIR / '05_aesrgan_results'
AESRGAN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_HALF_PRECISION = True
TILE_SIZE = 400
SUFFIX = 'aesrgan'

cmd = [
    sys.executable,
    str(AESRGAN_DIR / 'inference_aesrgan.py'),
    '--model_path', str(AESRGAN_MODEL_PATH),
    '--input', str(LAMA_OUTPUT_PATH),
    '--output', str(AESRGAN_OUTPUT_DIR),
    '--suffix', SUFFIX,
    '--tile', str(TILE_SIZE),
]

if USE_HALF_PRECISION:
    cmd.append('--half')

print('Ejecutando:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(AESRGAN_DIR), check=True)

aesrgan_output_path = AESRGAN_OUTPUT_DIR / f'{LAMA_OUTPUT_PATH.stem}_{SUFFIX}.png'
if not aesrgan_output_path.exists():
    generated = sorted(AESRGAN_OUTPUT_DIR.glob('*.png'))
    if not generated:
        raise FileNotFoundError(f'A-ESRGAN no generó PNGs en {AESRGAN_OUTPUT_DIR}')
    aesrgan_output_path = generated[-1]

img_aesrgan = open_rgb(aesrgan_output_path)
print(f'Resultado A-ESRGAN guardado en: {aesrgan_output_path}')

## 7. Comparativa final

In [ ]:
comparison_items = [
    ('Original/degradada', test_img),
    ('Máscara', test_mask),
    ('Entrada visual enmascarada', image_masked),
    ('LaMa', lama_result),
    ('LaMa + A-ESRGAN', img_aesrgan),
]

plt.figure(figsize=(24, 6))
for i, (title, image) in enumerate(comparison_items, start=1):
    plt.subplot(1, len(comparison_items), i)
    if image.mode == 'L':
        plt.imshow(image, cmap='gray')
    else:
        plt.imshow(image)
    plt.title(title)
    plt.axis('off')
plt.tight_layout()

COMPARISON_PATH = WORK_DIR / '06_comparativa_lama_aesrgan.png'
plt.savefig(COMPARISON_PATH, dpi=150, bbox_inches='tight')
plt.show()

print(f'Comparativa guardada en: {COMPARISON_PATH}')

## 8. Evaluación cuantitativa de resultados

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity


# Compatibilidad con nombres usados en distintas versiones del notebook
lama_img = lama_result if 'lama_result' in globals() else result
input_img = test_img
masked_img = image_masked
mask_img = test_mask
final_img = img_aesrgan


def list_image_files(folder):
    folder = Path(folder)
    valid_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}
    return sorted([p for p in folder.rglob('*') if p.suffix.lower() in valid_exts])


def find_clean_reference(input_path, clean_dir):
    input_path = Path(input_path)
    clean_dir = Path(clean_dir)
    clean_files = list_image_files(clean_dir)
    input_stem = input_path.stem.lower()

    # Estrategia 1: patrón 'imgNN' (p.ej. lrp_img52_... -> img52)
    match = re.search(r'img\d+', input_stem)
    if match:
        token = match.group(0)
        candidates = [p for p in clean_files if token in p.stem.lower()]
        if candidates:
            return candidates[0]

    # Estrategia 2: cualquier token numérico del nombre
    for num in re.findall(r'\d+', input_stem):
        candidates = [p for p in clean_files if num in p.stem]
        if candidates:
            return candidates[0]

    # Estrategia 3: prefijo alfanumérico común (primeros 6 chars)
    prefix = re.sub(r'[^a-z0-9]', '', input_stem)[:6]
    if prefix:
        candidates = [p for p in clean_files
                      if re.sub(r'[^a-z0-9]', '', p.stem.lower()).startswith(prefix)]
        if candidates:
            return candidates[0]

    return None


# Intenta recuperar la ruta de la imagen usada como entrada.
if 'INPUT_IMAGE_PATH' in globals():
    input_path_for_gt = Path(INPUT_IMAGE_PATH)
elif 'img_path' in globals():
    input_path_for_gt = Path(img_path)
else:
    input_path_for_gt = None


# Intenta encontrar automáticamente el GT limpio.
if 'GT_IMAGE_PATH' not in globals():
    if input_path_for_gt is not None and 'vintage_clean_path' in globals():
        GT_IMAGE_PATH = find_clean_reference(input_path_for_gt, vintage_clean_path)
    else:
        GT_IMAGE_PATH = None


# Si no lo encuentra, define esta ruta manualmente.
# GT_IMAGE_PATH = Path('/content/ruta/a/imagen_limpia.png')

if GT_IMAGE_PATH is None:
    raise FileNotFoundError(
        'No se encontró la imagen limpia de referencia. '
        'Define GT_IMAGE_PATH manualmente.'
    )

gt_img = open_rgb(GT_IMAGE_PATH)

print(f'Referencia limpia usada: {GT_IMAGE_PATH}')
print(f'Tamaño GT:              {gt_img.size}')
print(f'Tamaño entrada:         {input_img.size}')
print(f'Tamaño LaMa:            {lama_img.size}')
print(f'Tamaño final A-ESRGAN:  {final_img.size}')

In [ ]:
def compute_metrics(prediction, reference, mask=None, name=''):
    """
    Calcula métricas full-reference.

    La referencia se redimensiona al tamaño de la predicción.
    Esto permite evaluar:
    - LaMa a resolución original.
    - A-ESRGAN a resolución x4.
    """
    target_size = prediction.size

    pred = pil_rgb_to_array(prediction, size=target_size)
    ref = pil_rgb_to_array(reference, size=target_size)

    error = pred - ref

    mse = float(np.mean(error ** 2))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(error)))

    psnr = float(
        peak_signal_noise_ratio(
            ref,
            pred,
            data_range=255
        )
    )

    ssim, ssim_map = structural_similarity(
        ref,
        pred,
        channel_axis=2,
        data_range=255,
        full=True
    )
    ssim = float(ssim)

    row = {
        'experimento': name,
        'PSNR ↑': psnr,
        'SSIM ↑': ssim,
        'MAE ↓': mae,
        'RMSE ↓': rmse,
        'MSE ↓': mse,
    }

    if mask is not None:
        mask_bool = pil_mask_to_array(mask, size=target_size)

        if mask_bool.any():
            pred_m = pred[mask_bool]
            ref_m = ref[mask_bool]
            error_m = pred_m - ref_m

            mse_m = float(np.mean(error_m ** 2))
            rmse_m = float(np.sqrt(mse_m))
            mae_m = float(np.mean(np.abs(error_m)))

            if mse_m == 0:
                psnr_m = float('inf')
            else:
                psnr_m = float(20 * np.log10(255.0 / np.sqrt(mse_m)))

            ssim_map_2d = ssim_map.mean(axis=-1) if ssim_map.ndim == 3 else ssim_map
            ssim_m = float(ssim_map_2d[mask_bool].mean())

            row.update({
                'PSNR máscara ↑': psnr_m,
                'SSIM máscara ↑': ssim_m,
                'MAE máscara ↓': mae_m,
                'RMSE máscara ↓': rmse_m,
                'MSE máscara ↓': mse_m,
            })

    return row


def show_metrics_table(rows, title):
    df = pd.DataFrame(rows)

    columns = [
        'experimento',
        'PSNR ↑',
        'SSIM ↑',
        'MAE ↓',
        'RMSE ↓',
        'MSE ↓',
        'PSNR máscara ↑',
        'SSIM máscara ↑',
        'MAE máscara ↓',
        'RMSE máscara ↓',
        'MSE máscara ↓',
    ]

    df = df[[c for c in columns if c in df.columns]]

    print(title)
    display(df)

    return df

In [ ]:
inpainting_rows = [
    compute_metrics(
        prediction=masked_img,
        reference=gt_img,
        mask=mask_img,
        name='Entrada enmascarada'
    ),
    compute_metrics(
        prediction=lama_img,
        reference=gt_img,
        mask=mask_img,
        name='LaMa inpainting'
    ),
]

df_inpainting = show_metrics_table(
    inpainting_rows,
    'Evaluación del inpainting aislado'
)

In [ ]:
# Baseline clásico: la salida de LaMa reescalada al tamaño final con bicúbica.
lama_bicubic = lama_img.resize(
    final_img.size,
    Image.Resampling.BICUBIC
)

aesrgan_rows = [
    compute_metrics(
        prediction=lama_bicubic,
        reference=gt_img,
        mask=mask_img,
        name='LaMa + reescalado bicúbico'
    ),
    compute_metrics(
        prediction=final_img,
        reference=gt_img,
        mask=mask_img,
        name='LaMa + A-ESRGAN'
    ),
]

df_aesrgan_stage = show_metrics_table(
    aesrgan_rows,
    'Evaluación de la etapa A-ESRGAN'
)

delta_psnr = (
    df_aesrgan_stage.loc[df_aesrgan_stage['experimento'] == 'LaMa + A-ESRGAN', 'PSNR ↑'].values[0]
    -
    df_aesrgan_stage.loc[df_aesrgan_stage['experimento'] == 'LaMa + reescalado bicúbico', 'PSNR ↑'].values[0]
)

delta_ssim = (
    df_aesrgan_stage.loc[df_aesrgan_stage['experimento'] == 'LaMa + A-ESRGAN', 'SSIM ↑'].values[0]
    -
    df_aesrgan_stage.loc[df_aesrgan_stage['experimento'] == 'LaMa + reescalado bicúbico', 'SSIM ↑'].values[0]
)

print(f'Mejora PSNR de A-ESRGAN frente a bicúbica: {delta_psnr:.4f}')
print(f'Mejora SSIM de A-ESRGAN frente a bicúbica: {delta_ssim:.4f}')

In [ ]:
pipeline_rows = [
    compute_metrics(
        prediction=final_img,
        reference=gt_img,
        mask=mask_img,
        name='Pipeline completo: LaMa + A-ESRGAN'
    )
]

df_pipeline = show_metrics_table(
    pipeline_rows,
    'Evaluación del pipeline completo'
)

In [ ]:
plt.figure(figsize=(24, 8))

items = [
    ('GT limpio', gt_img),
    ('Entrada original', input_img),
    ('Entrada enmascarada', masked_img),
    ('LaMa inpainting', lama_img),
    ('LaMa bicúbico', lama_bicubic),
    ('LaMa + A-ESRGAN', final_img),
]

for i, (title, image) in enumerate(items, start=1):
    plt.subplot(1, len(items), i)
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')

plt.tight_layout()
plt.show()

### 8.1 LPIPS (Learned Perceptual Image Patch Similarity)

LPIPS mide similitud perceptual usando características internas de redes profundas (AlexNet). A diferencia de PSNR y SSIM —que operan píxel a píxel—, LPIPS captura diferencias percibidas por el sistema visual humano, siendo más representativo de la calidad visual real.

- **Valores menores son mejores** (↓).
- Se evalúa sobre la imagen completa y sobre la región enmascarada (bounding box de la máscara).
- Referencia: [Zhang et al., CVPR 2018 — The Unreasonable Effectiveness of Deep Features as a Perceptual Metric](https://github.com/richzhang/perceptualsimilarity).

In [ ]:
# lpips ya se instaló en la primera celda; los parches de compatibilidad
# (PIL._typing._Ink, basicsr/functional_tensor, ...) los aplicó aplicar_parches().
import torch
import lpips
from torchvision.transforms.functional import to_tensor as tvf_to_tensor

# Inicializar modelo LPIPS una sola vez (AlexNet: mejor para scoring, más rápido que VGG)
_LPIPS_FN = lpips.LPIPS(net='alex').eval()
if torch.cuda.is_available():
    _LPIPS_FN = _LPIPS_FN.cuda()

_LPIPS_MIN_CROP = 32  # tamaño mínimo de crop para que LPIPS sea estable


def _pil_to_lpips_tensor(image: Image.Image, size=None) -> torch.Tensor:
    """Convierte PIL RGB a tensor LPIPS normalizado [-1, 1], shape (1, 3, H, W)."""
    image = image.convert('RGB')
    if size is not None and image.size != size:
        image = image.resize(size, Image.Resampling.LANCZOS)
    t = tvf_to_tensor(image)              # [0, 1], shape (3, H, W)
    return (t * 2.0 - 1.0).unsqueeze(0)  # [-1, 1], shape (1, 3, H, W)


def _mask_bbox(mask: Image.Image, target_size=None) -> tuple | None:
    """Devuelve (left, top, right, bottom) del bounding box de la región enmascarada, o None."""
    if target_size is not None and mask.size != target_size:
        mask = mask.resize(target_size, Image.Resampling.NEAREST)
    arr = np.asarray(mask.convert('L'))
    rows = np.any(arr > 127, axis=1)
    cols = np.any(arr > 127, axis=0)
    if not rows.any():
        return None
    top    = int(np.where(rows)[0][0])
    bottom = int(np.where(rows)[0][-1]) + 1
    left   = int(np.where(cols)[0][0])
    right  = int(np.where(cols)[0][-1]) + 1
    return left, top, right, bottom


def compute_lpips_scores(
    prediction: Image.Image,
    reference: Image.Image,
    mask: Image.Image | None = None,
    name: str = '',
    fn: lpips.LPIPS = None,
) -> dict:
    """
    Calcula LPIPS global y, opcionalmente, sobre la región enmascarada (bounding box).
    La referencia se redimensiona al tamaño de la predicción para comparación justa.
    """
    fn = fn or _LPIPS_FN
    device = next(fn.parameters()).device
    target_size = prediction.size

    pred_t = _pil_to_lpips_tensor(prediction).to(device)
    ref_t  = _pil_to_lpips_tensor(reference, size=target_size).to(device)

    with torch.no_grad():
        lpips_global = float(fn(pred_t, ref_t).item())

    row = {'experimento': name, 'LPIPS ↓': lpips_global}

    if mask is not None:
        bbox = _mask_bbox(mask, target_size=target_size)
        if bbox is not None:
            left, top, right, bottom = bbox
            if (right - left) >= _LPIPS_MIN_CROP and (bottom - top) >= _LPIPS_MIN_CROP:
                ref_resized = reference.convert('RGB').resize(target_size, Image.Resampling.LANCZOS)
                pred_crop_t = _pil_to_lpips_tensor(prediction.crop(bbox)).to(device)
                ref_crop_t  = _pil_to_lpips_tensor(ref_resized.crop(bbox)).to(device)
                with torch.no_grad():
                    row['LPIPS máscara ↓'] = float(fn(pred_crop_t, ref_crop_t).item())
            else:
                print(f'[{name}] Crop de máscara demasiado pequeño, se omite LPIPS de máscara.')

    return row


# Calcular LPIPS para todos los experimentos
lpips_rows = [
    compute_lpips_scores(masked_img,   gt_img, mask=mask_img, name='Entrada enmascarada'),
    compute_lpips_scores(lama_img,     gt_img, mask=mask_img, name='LaMa inpainting'),
    compute_lpips_scores(lama_bicubic, gt_img, mask=mask_img, name='LaMa + reescalado bicúbico'),
    compute_lpips_scores(final_img,    gt_img, mask=mask_img, name='LaMa + A-ESRGAN'),
]

df_lpips = pd.DataFrame(lpips_rows)
print('LPIPS perceptual (↓ mejor)\n')
display(df_lpips)

# Guardar tabla
LPIPS_CSV_PATH = WORK_DIR / '07_lpips_scores.csv'
df_lpips.to_csv(LPIPS_CSV_PATH, index=False)
print(f'\nTabla guardada en: {LPIPS_CSV_PATH}')

# Visualización en barras
metric_cols = [c for c in ['LPIPS ↓', 'LPIPS máscara ↓'] if c in df_lpips.columns]
fig, axes = plt.subplots(1, len(metric_cols), figsize=(6 * len(metric_cols), 4))
if len(metric_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, metric_cols):
    bars = ax.bar(df_lpips['experimento'], df_lpips[col], color='steelblue', edgecolor='white')
    ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
    ax.set_title(col)
    ax.set_ylabel('LPIPS (↓ mejor)')
    ax.set_xticks(range(len(df_lpips)))
    ax.set_xticklabels(df_lpips['experimento'], rotation=15, ha='right')
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

LPIPS_FIG_PATH = WORK_DIR / '07_lpips_barplot.png'
plt.savefig(LPIPS_FIG_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada en: {LPIPS_FIG_PATH}')



## 9. Notas para siguientes experimentos

- Sustituir `auto_rectangle` por máscaras reales de daños físicos cuando se evalúen fotografías antiguas.
- Guardar una tabla de resultados por imagen: nombre, tamaño original, tamaño final, ruta de máscara, ruta LaMa y ruta A-ESRGAN.
- Añadir métricas cuantitativas solo cuando exista referencia limpia emparejada: PSNR, SSIM y, si procede, LPIPS.
- Separar esta prueba integrada del notebook de evaluación masiva sobre dataset.